# Data preparation of hydro data from JRC PECD

source: https://zenodo.org/record/3985078

detailed description of data in folder ../source_data/hydro_ENTSOE/

Creates the following parsed datasets

- capacities for entso-e base year (which one?) per technology and country
- Hourly ror generation per country year (1982-2017)
- Weekly storage inflows per country and year (1982-2017)
- Weekly reservoir levels per country and year (1982-2017) for setting storage start and end conditions

Cite as

De Felice, Matteo. (2020). ENTSO-E Hydropower modelling data (PECD) in CSV format (Version 4) [Data set]. Zenodo. http://doi.org/10.5281/zenodo.3985078

Updated by Jonas, 23.07.25

In [1]:
import pandas as pd
import wget
import os
from datetime import datetime as dt

c:\Users\jonas\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
#set if file should be downloaded again (yes/no)
download = "no"

In [3]:
dir_in = "../source_data/hydro_ENTSOE/"
dir_out = "../parsed_data/"
#fn = "PECD-hydro-capacities.csv"
fns = {"PECD-hydro-capacities.csv",
       "PECD-hydro-daily-ror-generation.csv",
       "PECD-hydro-weekly-inflows.csv",
       "PECD-hydro-weekly-reservoir-levels.csv"
      }

In [4]:
#file download
if download == "yes":
    for fn in fns:
        #os.remove(dir_in+fn)
        url = 'https://zenodo.org/records/3985078/files/'+fn+'?download=1'
        wget.download(url,dir_in+fn)    

In [5]:
#first we load a country dictionary
country_dict = pd.read_excel(dir_in+'hydro_ENTSOE_country_dict.xlsx')
country_dict = dict(country_dict.values.tolist())

# Capacities per technology

In [6]:
#next, load the ENTSO-E base capacities - we use them to calculate shares to distribute yearly values
df_capacities_zones = pd.read_csv(dir_in+'PECD-hydro-capacities.csv')
df_capacities_zones['tech'] = df_capacities_zones['type'] + ' - ' + df_capacities_zones['variable']
df_capacities = df_capacities_zones.copy()
df_capacities['zone'] = df_capacities['zone'].map(country_dict)
df_capacities = df_capacities.rename(columns={'zone':'country'})
df_capacities = df_capacities.groupby(['country','tech']).agg({
    'value':'sum'})
df_capacities = pd.pivot_table(df_capacities,index=['country'], columns='tech',values='value')
df_capacities.head()

tech,Pump Storage - Closed Loop - Cumulated (upper or head) reservoir capacity (GWh),Pump Storage - Closed Loop - Total pumping capacity (MW),Pump Storage - Closed Loop - Total turbining capacity (MW),Pump Storage - Open Loop - Cumulated (upper or head) reservoir capacity (GWh),Pump Storage - Open Loop - Total pumping capacity (MW),Pump Storage - Open Loop - Total turbining capacity (MW),Reservoir - Reservoir capacity (GWh),Reservoir - Total turbining capacity (MW),Run-of-River and pondage - Reservoir capacity linked to Run of River and Pondage units (GWh),Run-of-River and pondage - Total turbining capacity (MW)
country,,,,,,,,,,
AL,0.0,0.0,0.0,0.000,0.000,0.00,1450.0000,1652.190,0.000000,322.87
AT,0.0,0.0,0.0,1722.178,-2559.726,3459.28,762.3862,2429.554,16.764755,5782.00
BA,0.0,0.0,0.0,3.400,-440.000,440.00,1669.0000,521.000,26.100000,1009.00
BE,5.3,-1150.0,1308.0,0.000,0.000,0.00,0.0000,0.000,0.000000,113.94
BG,9.4,-784.0,864.0,255.300,-148.000,535.00,843.0000,1347.000,0.000000,462.00


In [7]:
#export to CSV
df_capacities.to_csv(dir_out + "hydro_capacities_base_ENTSO-E_adequacy.csv", encoding="utf-8")

# Hourly ror generation

In [8]:
#then, aggregate ror generation and calculate hourly values per day in MWh
df_ror_generation = pd.read_csv(dir_in+'PECD-hydro-daily-ror-generation.csv')
df_ror_generation['zone'] = df_ror_generation['zone'].map(country_dict)
df_ror_generation = df_ror_generation.rename(columns={'zone':'country'})
df_ror_generation['date'] = pd.to_datetime((df_ror_generation['year'].astype(str) 
                                            + '-' + df_ror_generation['Day'].astype(str)),
                                           format='%Y-%j')
df_ror_generation = df_ror_generation.groupby(['date','country']).agg({
    'Run of River Hydro Generation in GWh per day':'sum'})
df_ror_generation['RoR generation MWh per hour'] = df_ror_generation['Run of River Hydro Generation in GWh per day']/24*1000
df_ror_generation.head()

Run of River Hydro Generation in GWh per day  \
date       country                                                 
1982-01-01 AL                                           4.635531   
           AT                                          81.135479   
           BA                                          14.410000   
           BE                                           1.919660   
           BG                                           1.188977   

                    RoR generation MWh per hour  
date       country                               
1982-01-01 AL                        193.147119  
           AT                       3380.644955  
           BA                        600.416667  
           BE                         79.985843  
           BG                         49.540699

In [9]:
df_ror_generation.reset_index().country.unique()

array(['AL', 'AT', 'BA', 'BE', 'BG', 'CH', 'CZ', 'DE', 'ES', 'FI', 'FR',
       'GB', 'GR', 'HR', 'HU', 'IE', 'IT', 'LU', 'ME', 'MK', 'NO', 'PL',
       'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'TR'], dtype=object)

In [10]:
#export to CSV
df_ror_generation.to_csv(dir_out + "hydro_ror_generation_hourly_ENTSO-E_adequacy.csv", encoding="utf-8")

In [11]:
df_ror_generation[df_ror_generation.index.get_level_values('country')=='FI'].sum()

Run of River Hydro Generation in GWh per day    0.0
RoR generation MWh per hour                     0.0
dtype: float64

# Weekly storage inflows

In [12]:
#next, load weekly inflows to storage
df_inflows = pd.read_csv(dir_in+'PECD-hydro-weekly-inflows.csv')
df_inflows['zone'] = df_inflows['zone'].map(country_dict)
df_inflows = df_inflows.rename(columns={'zone':'country'})
#uncommented for now as this creates duplicates at end/beginning of year
#df_inflows['week'] = (df_inflows['week']-1).apply(lambda x: '{0:0>2}'.format(x))
#df_inflows['date'] = df_inflows['year'].astype(str) + '-W' + df_inflows['week'].astype(str) +'-1'
#df_inflows['date'] = pd.to_datetime((df_inflows['date']), format='%Y-W%W-%w')
df_inflows = df_inflows.groupby(['country','year','week']).agg({
    'Cumulated inflow into reservoirs per week in GWh':'sum',
    'Cumulated NATURAL inflow into the pump-storage reservoirs per week in GWh':'sum'})
df_inflows.head()

Cumulated inflow into reservoirs per week in GWh  \
country year week                                                     
AL      1982 1                                           160.138143   
             2                                           170.910241   
             3                                           138.513934   
             4                                           138.507779   
             5                                           118.381262   

                   Cumulated NATURAL inflow into the pump-storage reservoirs per week in GWh  
country year week                                                                             
AL      1982 1                                                   0.0                          
             2                                                   0.0                          
             3                                                   0.0                          
             4                                                   0.0                          
             5                                                   0.0

In [13]:
df_inflows.reset_index()[((df_inflows.reset_index().country == 'NO') 
                        )]

,country,year,week,Cumulated inflow into reservoirs per week in GWh,Cumulated NATURAL inflow into the pump-storage reservoirs per week in GWh
38099,NO,1982,1,0.0,1107.348094
38100,NO,1982,2,0.0,1592.940244
38101,NO,1982,3,0.0,1083.319961
38102,NO,1982,4,0.0,1642.438342
38103,NO,1982,5,0.0,822.780314
...,...,...,...,...,...
40002,NO,2017,49,0.0,0.000000
40003,NO,2017,50,0.0,0.000000
40004,NO,2017,51,0.0,0.000000
40005,NO,2017,52,0.0,0.000000


In [14]:
#export to CSV
df_inflows.to_csv(dir_out + "hydro_storage_inflows_weekly_ENTSO-E_adequacy.csv", encoding="utf-8")

# Weekly reservoir levels

In [15]:
#now, load weekly reservoir levels
df_levels_zones = pd.read_csv(dir_in+'PECD-hydro-weekly-reservoir-levels.csv')
df_levels_zones = df_levels_zones.rename(columns = {'Reservoir levels at beginning of each week (ratio) 0<=x<=1.0' : 'level'})
df_levels_zones = df_levels_zones.rename(columns = {'Minimum Reservoir levels at beginning of each week (ratio) 0<=x<=1.0' : 'minlevel'})
df_levels_zones = df_levels_zones.rename(columns = {'Maximum Reservoir level at beginning of each week (ratio) 0<=x<=1.0' : 'maxlevel'})
df_levels_zones['country'] = df_levels_zones['zone'].map(country_dict)
df_levels_zones.head()

,zone,week,year,minlevel,maxlevel,level,country
0,AT00,1,1982,0.244062,0.644739,NaN,AT
1,AT00,1,1983,0.244062,0.644739,NaN,AT
2,AT00,1,1984,0.244062,0.644739,NaN,AT
3,AT00,1,1985,0.244062,0.644739,NaN,AT
4,AT00,1,1986,0.244062,0.644739,NaN,AT


In [16]:
#we also load reservoir size per zone for calculation of per country weighted average 
df_capacities_zones = pd.pivot_table(df_capacities_zones,index=['zone'], columns='tech',values='value').reset_index()
df_capacities_zones['capacity_GWH_agg'] = df_capacities_zones['Pump Storage - Closed Loop - Cumulated (upper or head) reservoir capacity (GWh)'] + df_capacities_zones['Pump Storage - Open Loop - Cumulated (upper or head) reservoir capacity (GWh)'] + df_capacities_zones['Reservoir - Reservoir capacity (GWh)'] + df_capacities_zones['Run-of-River and pondage - Reservoir capacity linked to Run of River and Pondage units (GWh)']
df_capacities_zones['country'] = df_capacities_zones['zone'].map(country_dict)
df_capacities_zones = df_capacities_zones.reset_index()
#calculate capacity sum by country
df_capacities_zones_grouped = pd.DataFrame(df_capacities_zones.groupby(['country']).sum()['capacity_GWH_agg'])

In [17]:
#merge dfs and calculate and apply shares per zone
df_levels_zones_merge = df_levels_zones.merge(df_capacities_zones[['zone','capacity_GWH_agg']], on='zone')
df_levels_zones_merge2 = df_levels_zones_merge.merge(df_capacities_zones_grouped, on='country')
df_levels_zones_merge2['weight'] = df_levels_zones_merge2['capacity_GWH_agg_x'] / df_levels_zones_merge2['capacity_GWH_agg_y']
#FR is somehow missing so we set value to 1:
df_levels_zones_merge2['weight'][df_levels_zones_merge2['weight'] == 0] = 1
df_levels_zones_merge2['level_weighted'] = df_levels_zones_merge2['level'] * df_levels_zones_merge2['weight']
df_levels_zones_merge2['maxlevel_weighted'] = df_levels_zones_merge2['maxlevel'] * df_levels_zones_merge2['weight']
df_levels_zones_merge2['minlevel_weighted'] = df_levels_zones_merge2['minlevel'] * df_levels_zones_merge2['weight']

C:\Users\jonas\AppData\Local\Temp\ipykernel_36344\1573696192.py:6: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_levels_zones_merge2['weight'][df_levels_zones_merge2['weight'] == 0] = 1
C:\Users\jonas\AppData\Local\Temp\ipykernel_36344\15

In [18]:
#eventually, create country level sums
df_levels_weighted = df_levels_zones_merge2.groupby(['country','year','week']).sum()
df_levels_weighted.head(20)

zone  minlevel  maxlevel     level  capacity_GWH_agg_x  \
country year week                                                           
AL      1982 1     AL00       0.0       0.0  0.588966              1450.0   
        1983 1     AL00       0.0       0.0  0.588966              1450.0   
        1984 1     AL00       0.0       0.0  0.588966              1450.0   
        1985 1     AL00       0.0       0.0  0.588966              1450.0   
        1986 1     AL00       0.0       0.0  0.588966              1450.0   
        1987 1     AL00       0.0       0.0  0.588966              1450.0   
        1988 1     AL00       0.0       0.0  0.588966              1450.0   
        1989 1     AL00       0.0       0.0  0.588966              1450.0   
        1990 1     AL00       0.0       0.0  0.588966              1450.0   
        1991 1     AL00       0.0       0.0  0.588966              1450.0   
        1992 1     AL00       0.0       0.0  0.588966              1450.0   
        1993 1     AL00       0.0       0.0  0.588966              1450.0   
        1994 1     AL00       0.0       0.0  0.588966              1450.0   
        1995 1     AL00       0.0       0.0  0.588966              1450.0   
        1996 1     AL00       0.0       0.0  0.588966              1450.0   
        1997 1     AL00       0.0       0.0  0.588966              1450.0   
        1998 1     AL00       0.0       0.0  0.588966              1450.0   
        1999 1     AL00       0.0       0.0  0.588966              1450.0   
        2000 1     AL00       0.0       0.0  0.588966              1450.0   
        2001 1     AL00       0.0       0.0  0.588966              1450.0   

                   capacity_GWH_agg_y  weight  level_weighted  \
country year week                                               
AL      1982 1                 1450.0     1.0        0.588966   
        1983 1                 1450.0     1.0        0.588966   
        1984 1                 1450.0     1.0        0.588966   
        1985 1                 1450.0     1.0        0.588966   
        1986 1                 1450.0     1.0        0.588966   
        1987 1                 1450.0     1.0        0.588966   
        1988 1                 1450.0     1.0        0.588966   
        1989 1                 1450.0     1.0        0.588966   
        1990 1                 1450.0     1.0        0.588966   
        1991 1                 1450.0     1.0        0.588966   
        1992 1                 1450.0     1.0        0.588966   
        1993 1                 1450.0     1.0        0.588966   
        1994 1                 1450.0     1.0        0.588966   
        1995 1                 1450.0     1.0        0.588966   
        1996 1                 1450.0     1.0        0.588966   
        1997 1                 1450.0     1.0        0.588966   
        1998 1                 1450.0     1.0        0.588966   
        1999 1                 1450.0     1.0        0.588966   
        2000 1                 1450.0     1.0        0.588966   
        2001 1                 1450.0     1.0        0.588966   

                   maxlevel_weighted  minlevel_weighted  
country year week                                        
AL      1982 1                   0.0                0.0  
        1983 1                   0.0                0.0  
        1984 1                   0.0                0.0  
        1985 1                   0.0                0.0  
        1986 1                   0.0                0.0  
        1987 1                   0.0                0.0  
        1988 1                   0.0                0.0  
        1989 1                   0.0                0.0  
        1990 1                   0.0                0.0  
        1991 1                   0.0                0.0  
        1992 1                   0.0                0.0  
        1993 1                   0.0                0.0  
        1994 1                   0.0                0.0  
        1995 1         

In [19]:
#export to CSV
df_levels_weighted.to_csv(dir_out + "hydro_storage_levels_weekly_ENTSO-E_adequacy.csv", encoding="utf-8")

In [23]:
df_levels_weighted.index.get_level_values('country').unique()

Index(['AL', 'AT', 'CH', 'ES', 'FI', 'FR', 'PT', 'SE'], dtype='object', name='country')